# 21 — Validación en semanas nuevas frente al ingeniero

Se evalúan semanas posteriores al último objetivo usado en el informe, sólo cuando producción de la semana está cerrada. El cruce del ingeniero replica la suma por serie/origen/objetivo/horizonte y excluye líneas incompletas. El pronóstico se reentrena en cada origen con etiquetas cerradas antes de esa fecha. La selección climática es automática: usa las seis señales lagged si la cobertura train+test es ≥80%; de lo contrario, usa el mismo modelo sin clima. No imputa el clima no actualizado.

In [1]:
# Datos comunes y evaluación sin fuga temporal
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datos_analiticos_proyecto").is_dir())
DATA = ROOT / "Datos_analiticos_proyecto"
OUT = DATA / "experimentos_actualizacion_20261020"
OUT.mkdir(parents=True, exist_ok=True)
DEST = DATA / "modelo_con_proyecciones_teoricas"
K = ["finca", "producto", "color", "variedad"]
CASE = K + ["origen", "semana_objetivo", "horizonte"]

# Features calculadas al origen. y se sustituye por la producción 2026 recién limpia,
# pero las entradas históricas del caso quedan congeladas como estaban al emitir.
base = pd.read_parquet(DEST / "base_analitica_proyecciones.parquet")
real26 = pd.read_parquet(DATA / "produccion_real_2026_actualizada.parquet")
real26 = real26.loc[real26.finca.eq("GAITANA") & real26.llave_completa].copy()
real_semanal = real26.groupby(K + ["semana_inicio"], as_index=False).agg(
    y_nuevo=("tallos", "sum"), ultimo_dia_real=("fecha_corte", "max"))
ultima_fecha = real26.fecha_corte.max()
ultima_semana = ultima_fecha - pd.Timedelta(days=ultima_fecha.weekday())
ultima_semana_completa = ultima_semana if ultima_fecha.weekday() == 6 else ultima_semana - pd.Timedelta(days=7)

casos = base.loc[base.finca.eq("GAITANA")].copy()
casos = casos.merge(real_semanal, left_on=K + ["semana_objetivo"],
                    right_on=K + ["semana_inicio"], how="left", validate="many_to_one")
casos["y_eval"] = casos.y_nuevo.where(casos.y_nuevo.notna(), casos.y)
casos["admisible_actual"] = casos.media_4.notna() & casos.observaciones_previas.ge(26)
casos["horizonte"] = casos.horizonte.astype(int)
model_data = casos.loc[casos.y_eval.notna() & casos.admisible_actual].copy()

# Mismo cruce nominal estricto del benchmark oficial: suma las líneas de la misma
# serie/origen/objetivo/horizonte y excluye cantidades incompletas y envíos posteriores.
e = pd.read_parquet(DATA / "estimados_semanales_limpio.parquet")
e = e.loc[e.finca.eq("GAITANA") & e.en_h1_h5 & ~e.es_ajustado
          & e.origen_solicitado.dt.year.eq(2026) & e.semana_inicio.dt.year.eq(2026)].copy()
e = e.rename(columns={"origen_solicitado": "origen", "semana_inicio": "semana_objetivo"})
engineer = e.groupby(CASE, dropna=False).agg(
    ingeniero=("estimado", "sum"), lineas=("estimado", "size"),
    cantidades=("estimado", "count"), archivos=("archivo_origen", "nunique"),
    posterior=("archivo_es_posterior", "max")).reset_index()
engineer["ingeniero_completo"] = engineer.lineas.eq(engineer.cantidades) & engineer.archivos.eq(1) & ~engineer.posterior.fillna(False)
casos = casos.merge(engineer, on=CASE, how="inner", validate="one_to_one")
casos = casos.loc[casos.ingeniero_completo & casos.y_eval.notna() & casos.admisible_actual].copy()
casos = casos.loc[casos.semana_objetivo.gt(pd.Timestamp("2026-08-31"))
                  & casos.semana_objetivo.le(ultima_semana_completa)].copy()
print("Última fecha real:", ultima_fecha.date(), "| última semana completa:", ultima_semana_completa.date())
print("Casos nuevos comparables tras cruce estricto:", len(casos))
print("Semanas objetivo:", sorted(casos.semana_objetivo.dt.strftime("%Y-%m-%d").unique()))
print("Clima disponible hasta:", pd.to_datetime(pd.read_parquet(DATA / "clima_limpio.parquet").fecha).max().date())


Última fecha real: 2026-10-03 | última semana completa: 2026-09-21
Casos nuevos comparables tras cruce estricto: 1226
Semanas objetivo: ['2026-09-07', '2026-09-14', '2026-09-21']
Clima disponible hasta: 2026-09-21


In [2]:
CATS = ["producto", "color", "variedad"]
MEMORY = ["produccion_lag1", "produccion_lag2", "produccion_lag3", "media_4", "media_8",
          "desv_4", "tendencia_4_8", "cambio_reciente", "produccion_52"]
CLIMATE = ["temperatura_semana_lag1", "temperatura_semana_media4",
           "humedad_semana_lag1", "humedad_semana_media4",
           "radiacion_semana_lag1", "radiacion_semana_media4"]
THEORY = ["teoria_modelo", "antiguedad_curva", "desviacion_reciente_teoria",
          "teoria_previa_1", "teoria_previa_2", "revision_teoria"]
BASE_X = CATS + ["horizonte", "semana_seno", "semana_coseno"] + MEMORY

def metricas(g, p):
    y = np.asarray(g.y_eval, dtype=float); p = np.asarray(p, dtype=float)
    den = y.sum()
    return {"n": len(y), "MAE_tallos": float(np.mean(np.abs(y-p))),
            "WAPE_pct": float(100*np.abs(y-p).sum()/den) if den else np.nan,
            "sesgo_pct": float(100*(p-y).sum()/den) if den else np.nan}

def elegir_clima(train, test, habilitar=True, minimo=0.80):
    """Usa clima sólo si existe cobertura suficiente en el ajuste evaluado."""
    if not habilitar or not all(c in train and c in test for c in CLIMATE):
        return [], 0.0, "sin clima: columnas no disponibles"
    cov_train = float(train[CLIMATE].notna().mean().mean())
    latest_origin = test.origen.max()
    current = test.loc[test.origen.eq(latest_origin), CLIMATE]
    cov_current = float(current.notna().mean().mean()) if len(current) else 0.0
    cobertura = min(cov_train, cov_current)
    if cobertura < minimo:
        return [], cobertura, f"sin clima: cobertura {cobertura:.1%} < {minimo:.0%}"
    return CLIMATE, cobertura, f"con clima: cobertura {cobertura:.1%}"

def crear_modelo(cols, params=None):
    params = params or {}
    cats = [c for c in CATS if c in cols]
    nums = [c for c in cols if c not in cats]
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cats),
        ("num", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), nums),
    ])
    reg = HistGradientBoostingRegressor(
        learning_rate=params.get("learning_rate", .08), max_iter=params.get("max_iter", 80),
        max_leaf_nodes=params.get("max_leaf_nodes", 15), min_samples_leaf=40,
        l2_regularization=1.0, early_stopping=False, random_state=42)
    return make_pipeline(prep, reg)

def pronosticar_ruta(train, test, cols, params=None):
    """Ajusta la media4 y, con soporte suficiente, la teoría completa por separado."""
    train = train.loc[train.y_eval.notna() & train.media_4.notna()].copy()
    pred = pd.Series(index=test.index, dtype=float)
    if train.empty or test.empty: return pred
    with threadpool_limits(limits=2):
        fallback = crear_modelo(cols, params)
        fallback.fit(train[cols], train.y_eval-train.media_4)
        pred.loc[test.index] = np.maximum(0, test.media_4 + fallback.predict(test[cols]))
        tr = train.loc[train.curva_completa.fillna(False) & train.teoria_modelo.notna()]
        te = test.loc[test.curva_completa.fillna(False) & test.teoria_modelo.notna()]
        if len(tr) >= 40 and len(te):
            use = [c for c in cols + THEORY if c in train.columns]
            teoria = crear_modelo(use, params)
            teoria.fit(tr[use], tr.y_eval-tr.teoria_modelo)
            pred.loc[te.index] = np.maximum(0, te.teoria_modelo + teoria.predict(te[use]))
    return pred

def fuera_producto(df, patron):
    return df.producto.isin(str(patron).split("|"))

def evaluar_origenes(data, grupos="agrupado", excluir_entrenamiento=None, clima="auto", params=None, train_pool=None):
    """Reentrena sólo con semanas reales cerradas antes de cada origen evaluado."""
    train_pool = model_data if train_pool is None else train_pool
    salidas=[]; controles=[]
    for origen, test in data.groupby("origen", sort=True):
        test = test.copy()
        train = train_pool.loc[train_pool.semana_objetivo + pd.Timedelta(days=7) <= origen].copy()
        if excluir_entrenamiento:
            train = train.loc[~fuera_producto(train, excluir_entrenamiento)]
        if train.empty: continue
        climat, cov, modo = elegir_clima(train, test, habilitar=(clima != "sin"))
        x = BASE_X + climat
        test["prediccion"] = np.nan
        grp = test.horizonte if grupos == "horizonte" else np.where(test.horizonte <= 2, "H1_H2", "H3_H5")
        test["_grp"] = grp
        train["_grp"] = train.horizonte if grupos == "horizonte" else np.where(train.horizonte <= 2, "H1_H2", "H3_H5")
        for gval, te in test.groupby("_grp"):
            tr = train.loc[train._grp.eq(gval)]
            if tr.empty: continue
            teorio = x + THEORY
            te["prediccion"] = pronosticar_ruta(tr, te, x)
            test.loc[te.index, "prediccion"] = te.prediccion
            if test.loc[te.index, "prediccion"].notna().any():
                controles.append({"origen": origen, "grupo": str(gval), "train_n": len(tr),
                                  "test_n": len(te), "clima": modo, "cobertura_clima": cov})
        test.drop(columns="_grp", inplace=True)
        salidas.append(test)
    if not salidas: return pd.DataFrame(), pd.DataFrame(controles)
    return pd.concat(salidas).dropna(subset=["prediccion"]), pd.DataFrame(controles)

def resumen_comparacion(pred):
    rows=[]
    for nombre,col in [("Modelo", "prediccion"),("Ingeniero","ingeniero"),("Media 4 semanas","media_4"),("Teoría aplicable","teoria_modelo")]:
        g=pred.loc[pred[col].notna()].copy()
        if len(g): rows.append({"metodo":nombre,**metricas(g,g[col])})
    return pd.DataFrame(rows)


In [3]:
# Evaluación nueva y comparación estricta
pred, control = evaluar_origenes(casos, grupos="agrupado", clima="auto", train_pool=model_data)
if pred.empty:
    raise ValueError("No hay casos nuevos comparables y admisibles; revisar fechas y llaves.")
resumen = resumen_comparacion(pred)
por_h = pd.DataFrame([{"horizonte":h, **resumen_comparacion(g).set_index("metodo").to_dict(orient="index").get("Modelo", {})}
                      for h,g in pred.groupby("horizonte")])
display(resumen); display(por_h); display(control)
print("Casos idénticos entre modelo e ingeniero:",len(pred),"| semanas:",pred.semana_objetivo.nunique())
pred.to_parquet(OUT/"validacion_semanas_nuevas.parquet",index=False)
resumen.to_csv(OUT/"resumen_validacion_nueva.csv",index=False)
control.to_csv(OUT/"control_clima_validacion_nueva.csv",index=False)


,metodo,n,MAE_tallos,WAPE_pct,sesgo_pct
0,Modelo,1226,2223.926253,17.541753,-5.067134
1,Ingeniero,1226,2072.071778,16.343964,-7.507109
2,Media 4 semanas,1226,2349.011623,18.528394,-5.629584
3,Teoría aplicable,93,2229.741262,32.187584,1.529015


,horizonte,n,MAE_tallos,WAPE_pct,sesgo_pct
0,1,103,1593.580565,12.984493,4.794710
1,2,205,1833.770188,15.007016,2.756645
2,3,306,2167.042761,16.896068,-7.401120
3,4,306,2400.477693,18.716121,-7.314220
4,5,306,2577.812550,20.098771,-8.656151


,origen,grupo,train_n,test_n,clima,cobertura_clima
0,2026-08-10,H3_H5,83954,103,con clima: cobertura 100.0%,1.0
1,2026-08-17,H3_H5,84297,205,con clima: cobertura 100.0%,1.0
2,2026-08-24,H3_H5,84644,306,con clima: cobertura 100.0%,1.0
3,2026-08-31,H1_H2,58321,103,con clima: cobertura 100.0%,1.0
4,2026-08-31,H3_H5,84991,203,con clima: cobertura 100.0%,1.0
5,2026-09-07,H1_H2,58556,205,con clima: cobertura 100.0%,1.0
6,2026-09-07,H3_H5,85341,101,con clima: cobertura 100.0%,1.0


Casos idénticos entre modelo e ingeniero: 1226 | semanas: 3
